# 01 · Platform Setup
Shared helpers used by every Banana DS notebook.
Run all cells top-to-bottom once per session; subsequent notebooks `import`
from the globals set here.

| Variable | Purpose |
|---|---|
| `API_BASE` | Root URL for all API calls |
| `call_endpoint` | Low-level HTTP helper |
| `get` / `post` / `patch` | Convenience wrappers |
| `pp` | Pretty-print JSON |
| `ENDPOINTS` | Live endpoint catalog loaded from OpenAPI |

In [ ]:
# ── Core imports & config ──────────────────────────────────────────────────
import json, urllib.request, urllib.error

API_BASE = 'https://api.banana.engineer'

def pp(obj):
    """Pretty-print a dict or raw JSON string."""
    if isinstance(obj, str):
        try: obj = json.loads(obj)
        except Exception: print(obj); return
    print(json.dumps(obj, indent=2))

print('config_ok | base:', API_BASE)

In [ ]:
# ── HTTP helper ────────────────────────────────────────────────────────────
def call_endpoint(method, path, payload=None, headers=None):
    url = f"{API_BASE}{path}"
    h = {'Accept': 'application/json'}
    if headers: h.update(headers)
    data = None
    if payload is not None:
        h['Content-Type'] = 'application/json'
        data = json.dumps(payload).encode()
    req = urllib.request.Request(url, data=data, headers=h, method=method.upper())
    try:
        with urllib.request.urlopen(req) as r:
            return {'ok': True, 'status': r.status, 'body': r.read().decode('utf-8', errors='replace')}
    except urllib.error.HTTPError as e:
        return {'ok': False, 'status': e.code, 'body': e.read().decode('utf-8', errors='replace')}
    except Exception as ex:
        return {'ok': False, 'status': None, 'body': str(ex)}

def get(path, **kw):    return call_endpoint('GET',    path, **kw)
def post(path, **kw):   return call_endpoint('POST',   path, **kw)
def patch(path, **kw):  return call_endpoint('PATCH',  path, **kw)
def delete(path, **kw): return call_endpoint('DELETE', path, **kw)

print('helpers_ready')

In [ ]:
# ── Load live endpoint catalog from OpenAPI ────────────────────────────────
r = get('/swagger/v1/swagger.json')
if not r['ok']:
    print('catalog_error:', r['status'], r['body'][:200])
    ENDPOINTS = []
else:
    spec = json.loads(r['body'])
    ENDPOINTS = []
    VERBS = {'get','post','put','patch','delete','head','options'}
    for path, methods in (spec.get('paths') or {}).items():
        for verb, details in methods.items():
            if verb.lower() not in VERBS: continue
            ENDPOINTS.append({
                'method':  verb.upper(),
                'path':    path,
                'summary': details.get('summary') or details.get('operationId',''),
                'group':   (details.get('tags') or ['Ungrouped'])[0],
            })
    ENDPOINTS.sort(key=lambda e: (e['group'], e['path'], e['method']))
    print(f'catalog_loaded | endpoints: {len(ENDPOINTS)}')
    groups = {}
    for e in ENDPOINTS:
        groups[e['group']] = groups.get(e['group'], 0) + 1
    for g in sorted(groups):
        print(f'  {g}: {groups[g]}')

In [ ]:
# ── Platform health smoke ─────────────────────────────────────────────────
r = get('/health')
pp({'health_status': r['status'], 'body': r['body'][:300]})